# DYNAMIC DB WITH CHROMA AND HUGGING FACE

In [1]:
import os

with open("hf_token.txt", "r") as f:
    HF_TOKEN = f.readline().strip()

os.environ["HF_TOKEN"] = HF_TOKEN 

In [2]:
from transformers import AutoTokenizer
import transformers
import torch
model = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model)

In [3]:
pipeline = transformers.pipeline(
    "text-generation",
    model = model,
    torch_dtype=torch.float16,
    device_map = "auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [5]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("sciq", split = "train")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.99M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/339k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/343k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11679 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
filtered_dataset = dataset.filter(lambda x: x["support"] != "" and x["correct_answer"] != "")
print("Number of questions with support: ", len(filtered_dataset))

Filter:   0%|          | 0/11679 [00:00<?, ? examples/s]

Number of questions with support:  10481


In [8]:
df = pd.DataFrame(filtered_dataset)
columns_to_drop = ["distractor3", "distractor2", "distractor1"]
df.drop(columns=columns_to_drop, inplace=True)

In [9]:
df["completion"] = df["correct_answer"] + "because " + df["support"]
df.dropna(subset=["completion"], inplace=True)
df

,question,correct_answer,support,completion
0,What type of organism is commonly used in prep...,mesophilic organisms,"Mesophiles grow best in moderate temperature, ...",mesophilic organismsbecause Mesophiles grow be...
1,What phenomenon makes global winds blow northe...,coriolis effect,Without Coriolis Effect the global winds would...,coriolis effectbecause Without Coriolis Effect...
2,Changes from a less-ordered state to a more-or...,exothermic,Summary Changes of state are examples of phase...,exothermicbecause Summary Changes of state are...
3,What is the least dangerous radioactive decay?,alpha decay,All radioactive decay is dangerous to living t...,alpha decaybecause All radioactive decay is da...
4,Kilauea in hawaii is the world’s most continuo...,smoke and ash,Example 3.5 Calculating Projectile Motion: Hot...,smoke and ashbecause Example 3.5 Calculating P...
...,...,...,...,...
10476,The enzyme pepsin plays an important role in t...,peptides,Protein A large part of protein digestion take...,peptidesbecause Protein A large part of protei...
10477,What remains a constant of radioactive substan...,rate of decay,The rate of decay of a radioactive substance i...,rate of decaybecause The rate of decay of a ra...
10478,"Terrestrial ecosystems, also known for their d...",biomes,"Terrestrial ecosystems, also known for their d...","biomesbecause Terrestrial ecosystems, also kno..."
10479,High explosives create shock waves that exceed...,supersonic,The modern day formulation of gun powder is ca...,supersonicbecause The modern day formulation o...


In [10]:
df.shape

(10481, 4)

In [11]:
print(df.columns)

Index(['question', 'correct_answer', 'support', 'completion'], dtype='object')


In [14]:
import chromadb
client = chromadb.Client()
collection_name = "sciq_support"

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [17]:
collections = client.list_collections()
if not any(collection_name == col.name for col in collections):
    collection = client.create_collection(collection_name)

results = collection.get()
for result in results:
    print(result)

ids
embeddings
documents
uris
included
data
metadatas


In [26]:
import time
embedding_model = "all-MiniLM-L6-v2" #distilled from BERT

len_df = len(df)
start_time = time.time()
completion_list = df["completion"][:].astype(str).tolist()

collections.add(
    ids = [str(i) for i in range(len_df)],
    documents = comopletion_list,
    metadatas = [{"type": "completions"} for _ in range(len_df)],
)

response_time = time.time() - start_time 
print(f"Data successfully inserted in chromadb collection {collection_name} in {response_time} s") 

AttributeError: 'list' object has no attribute 'add'